# Malware - prepare the demo tables

**Input** - the UCI [NATICUSdroid Android permissions](https://archive.ics.uci.edu/dataset/722/naticusdroid+android+permissions+dataset) archive, downloaded once to `data/malware/raw/uci722/` and read from there afterwards.

**Output** - at the paths `configs/malware.yaml` declares:

| File | |
| --- | --- |
| `data/malware/train.csv` · `valid.csv` · `test.csv` | the fixed 60 / 20 / 20 split the pipeline reads |
| `data/malware/few_shot.csv` | the example rows a discovery run shows: 10 batches of 32, same columns as `train.csv` plus `batch` |
| `data/malware/screen_train.csv` · `screen_valid.csv` | the rows discovery's screen fits and scores on, same format as `train.csv` |
| `data/malware/column_mapping.csv` | sanitized name → the archive's name |
| `data/malware/column_descriptions.json` | the fully qualified permission names, for discovery |

**The data** - 29,332 Android apps (2010-2019), 86 permission flags, 50.1% malware. It is the balanced counterpart to bankruptcy's 3%, and every feature is a 0/1 flag - did the app's manifest request this permission - so there is no arithmetic to do on one column. A useful proposal has to *combine* flags: how many permissions of some risky kind, whether two common ones occur together.

**The scenario** - the incumbent model uses the 58 standard AOSP permissions (`android.permission.*`). The candidates are the 28 **vendor and third-party** permissions - launcher badge counts, in-app billing, push messaging - which an analyst would have to collect and normalise across manufacturers. Whether they add anything decides whether that work is worth doing.

The family carries its own redundancy control, with nothing planted: six are launcher badge-count permissions, and an app that wants a badge requests every vendor's variant at once, so they correlate at |rho| 0.97-0.999 and the redundancy screen should drop them as a group.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# The repo root, wherever this notebook is run from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from preprocessing import (
    build_sample,
    build_shot_batches,
    fetch_archive,
    resolve_path,
    sanitize_columns,
    stratified_split,
    write_splits,
)
from validation import load_config
from validation.data import resolve_features

pd.set_option("display.width", 160, "display.max_columns", 12)

In [2]:
# Input: the UCI archive, cached here after the first download.
URL = ("https://archive.ics.uci.edu/static/public/722/"
       "naticusdroid+android+permissions+dataset.zip")
RAW_DIR = ROOT / "data" / "malware" / "raw" / "uci722"

# Output: written to the paths this config declares, and checked against it.
CONFIG = ROOT / "configs" / "malware.yaml"

SEED = 42                          # the split and the clustering
VALID_SIZE, TEST_SIZE = 0.2, 0.2   # 60 / 20 / 20
SHOTS, SHOT_BATCHES = 32, 10       # example rows per batch; one batch per discovery round
SCREEN_SIZE, SCREEN_BALANCE = {"train": 3000, "valid": 1000}, False   # the screen rows; already ~50/50

cfg = load_config(CONFIG)
target, id_col = cfg.data.target, cfg.data.id_cols[0]

## 1. Load the archive

The headers are the permission names themselves (`android.permission.GET_ACCOUNTS`). Sanitized to snake_case for use as columns; the originals are kept in `mapping` and become the descriptions - exactly what a proposer needs to know what a permission grants.

In [3]:
raw = pd.read_csv(fetch_archive(URL, RAW_DIR, "data.csv"))
frame, mapping = sanitize_columns(raw)
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
mapping.head()

29,332 rows x 87 columns


,original,column
0,android.permission.GET_ACCOUNTS,android_permission_get_accounts
1,com.sonyericsson.home.permission.BROADCAST_BADGE,com_sonyericsson_home_permission_broadcast_badge
2,android.permission.READ_PROFILE,android_permission_read_profile
3,android.permission.MANAGE_ACCOUNTS,android_permission_manage_accounts
4,android.permission.WRITE_SYNC_SETTINGS,android_permission_write_sync_settings


## 2. Shape the table

Rename the target, confirm every feature really is a flag, and derive the candidate family from the archive's own names: everything outside the `android.permission.` namespace. The config declares the same list, and the two must agree - disagreement means one of them has drifted.

In [4]:
frame = frame.rename(columns={"result": target})
mapping.loc[mapping["column"] == "result", "column"] = target

features = [c for c in frame.columns if c != target]
non_binary = [c for c in features if set(frame[c].dropna().unique()) - {0, 1}]
assert not non_binary, f"expected every feature to be a 0/1 flag; these are not: {non_binary[:5]}"

family = [row.column for row in mapping.itertuples()
          if row.column != target and not str(row.original).startswith("android.permission.")]
if cfg.features.new:
    assert sorted(family) == sorted(cfg.features.new), (
        f"in the config only: {sorted(set(cfg.features.new) - set(family))}; "
        f"in the archive only: {sorted(set(family) - set(cfg.features.new))}")

frame.insert(0, id_col, np.arange(len(frame)))
print(f"{len(features) - len(family)} standard AOSP incumbents + {len(family)} vendor / third-party candidates")

58 standard AOSP incumbents + 28 vendor / third-party candidates


## 3. Profile

No missing cells and no sentinel codes. What matters for a flag table is how often each permission is requested: one no app in train ever asks for is constant, and the data quality gate drops it.

In [5]:
features = [c for c in frame.columns if c not in {target, id_col}]
profile = pd.DataFrame({
    "dtype": frame[features].dtypes.astype(str),
    "n_unique": frame[features].nunique(),
    "missing_rate": frame[features].isna().mean().round(4),
})
print(f"{len(frame):,} rows, {len(features)} features")
print(f"target {target!r}: {frame[target].mean():.2%} positive "
      f"({int(frame[target].sum()):,} of {len(frame):,})")
print(f"missing: {frame[features].isna().to_numpy().mean():.1%} of feature cells, "
      f"{int((profile['missing_rate'] > 0).sum())} of {len(features)} columns affected")
rates = frame[features].mean().sort_values()
pd.concat([rates.head(4), rates.tail(4)]).rename("request_rate").round(3).to_frame()

29,332 rows, 86 features
target 'is_malware': 50.12% positive (14,700 of 29,332)
missing: 0.0% of feature cells, 0 of 86 columns affected


,request_rate
android_permission_android_permission_read_phone_state,0.004
android_permission_process_incoming_calls,0.004
android_permission_foreground_service,0.004
com_sec_android_iap_permission_billing,0.004
android_permission_read_phone_state,0.587
android_permission_write_external_storage,0.669
android_permission_access_network_state,0.949
android_permission_internet,0.975


## 4. Split 60 / 20 / 20

Stratified on the target, so each part keeps the base rate, with a fixed seed. This is the **only** split: the pipeline reads these three tables exactly as written and never re-splits. Discovery draws on train and valid only - the few-shot rows the proposer sees come from train, and its screen fits on train and scores on valid - so test is touched by nothing before the final verdict.

In [6]:
frames = stratified_split(frame, target, valid_size=VALID_SIZE, test_size=TEST_SIZE, seed=SEED)

pd.DataFrame({
    name: {"rows": len(part), "positives": int(part[target].sum()),
           "positive_rate": round(part[target].mean(), 4)}
    for name, part in frames.items()
}).T

,rows,positives,positive_rate
train,17600.0,8820.0,0.5011
valid,5866.0,2940.0,0.5012
test,5866.0,2940.0,0.5012


## 5. Few-shot example rows

The rows a discovery run prints under every column of its prompt - the only concrete data the proposer ever sees. They are chosen per class by KMeans on train's incumbent columns, one row per cluster (16 per class), so they cover the table rather than its densest region. Batch *r* is shown in round *r*; batch *b* takes the *b*-th closest row of each cluster, so successive rounds see different rows from the same regions.

Saved as the rows themselves - the same columns as `train.csv`, plus `batch` - and read back and cleaned exactly as the splits are, so the prompt shows what the models see.

Every column is a coded flag (`continuous_columns: []` in the config), so the clustering one-hot encodes them and the prompt shows levels, never ranges.

In [7]:
base, new = resolve_features(frames["train"], cfg.features, cfg.data)
continuous = cfg.discovery.continuous_columns
categorical = ([c for c in base if c not in set(continuous)] if continuous is not None
               else [c for c in cfg.discovery.categorical_columns if c in base])

batches = build_shot_batches(frames["train"], target, columns=base, categorical=categorical,
                             shots=SHOTS, batches=SHOT_BATCHES, seed=SEED)
few_shot = pd.concat([b.assign(batch=i) for i, b in enumerate(batches)], ignore_index=True)
few_shot = few_shot[["batch", *frames["train"].columns]]   # train.csv's columns, plus batch
assert few_shot[id_col].isin(frames["train"][id_col]).all()

print(f"{len(base)} incumbent columns ({len(categorical)} shown as coded categories), "
      f"{len(new)} candidates")
print(f"{len(batches)} batches x {len(batches[0])} rows, from {len(frames['train']):,} train rows")
few_shot.groupby("batch")[target].agg(rows="size", positives="sum").T

58 incumbent columns (58 shown as coded categories), 28 candidates
10 batches x 32 rows, from 17,600 train rows


batch,0,1,2,3,4,5,6,7,8,9
rows,32,32,32,32,32,32,32,32,32,32
positives,16,16,16,16,16,16,16,16,16,16


## 6. Screen sample

The rows discovery's screen fits each proposal on - a sample of **train** - and scores it on - a sample of **valid**. It is the cheap signal the proposer gets back every round; the decision is made later, on the full splits. When `SCREEN_BALANCE` is true each class contributes up to half the rows, so a rare class is kept whole instead of the handful a uniform draw would give. Saved as the rows themselves, in the same format as `train.csv`.

Scoring on valid means the proposer's feedback comes from valid rows, so valid stops being an independent check on what it proposes; test still is. To keep valid independent, draw both samples from disjoint rows of train. `discovery.screen_data: splits` in the config skips these files and uses the whole train and valid splits.

In [8]:
screen = {
    part: build_sample(frames[part], target, SCREEN_SIZE[part],
                       balance=SCREEN_BALANCE, seed=SEED)
    for part in ("train", "valid")
}

pd.DataFrame({
    part: {"rows": len(rows), "positives": int(rows[target].sum()),
           "positive_rate": round(rows[target].mean(), 4)}
    for part, rows in screen.items()
}).T

,rows,positives,positive_rate
train,3000.0,1472.0,0.4907
valid,1000.0,505.0,0.5050


## 7. Check against the config, then write

`write_splits` refuses to write anything if the tables disagree with `configs/malware.yaml`: a candidate the config names but the table lacks, a missing or non-binary target, splits with different columns, or an id in two splits.

In [9]:
written = write_splits(cfg, frames, root=ROOT, mapping=mapping)
written["few_shot"] = resolve_path(cfg.discovery.few_shot_path, ROOT)
few_shot.to_csv(written["few_shot"], index=False)
for part, rows in screen.items():
    written[f"screen_{part}"] = resolve_path(cfg.discovery.screen_paths[part], ROOT)
    rows.to_csv(written[f"screen_{part}"], index=False)

for name, path in written.items():
    print(f"{name:<13} {path.relative_to(ROOT)}")

train         data/malware/train.csv
valid         data/malware/valid.csv
test          data/malware/test.csv
descriptions  data/malware/column_descriptions.json
mapping       data/malware/column_mapping.csv
few_shot      data/malware/few_shot.csv
screen_train  data/malware/screen_train.csv
screen_valid  data/malware/screen_valid.csv


Next, from the repo root:

```bash
autofe -c configs/malware.yaml                                  # the declared candidates
autofe -c configs/malware.yaml --set discovery.enabled=true     # plus LLM-proposed ones
```